# 07 — Human TL;DR vs. AI baseline, by community

The AI summary is a fixed REFERENCE POINT ("what a plain summary looks like").
For every post we measure the same four things on the human TL;DR and on the
AI summary, then compare per subreddit and overall:

  ① first-person density        (voice — does "I" survive?)
  ② surface-flag rates          (share with '?', share with advice words) — rates only
  ③ cosine to the post          (semantic distance)
  ④ keyword containment         (lexical overlap with the post)

We do NOT subtract "human minus AI" as if the human were ground truth; we
place the two side by side. Read the AI column as a baseline, not a target.

Needs sample.jsonl and ai_summaries.jsonl. Uses cosine.parquet from nb 06 if
present, otherwise computes cosine here.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1] if "__file__" in dir() else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from tldr_audit.features import features_for_post  # noqa: E402
from tldr_audit.semantic import pairwise_cosine  # noqa: E402

FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
COLORS = {"human": "#333333", "ai": "#D55E00"}

sample_path = ROOT / "data" / "interim" / "sample.jsonl"
ai_path = ROOT / "data" / "interim" / "ai_summaries.jsonl"
if not ai_path.exists():
    raise FileNotFoundError(
        "data/interim/ai_summaries.jsonl not found — run scripts/04_ai_baseline.py first."
    )

posts = {json.loads(l)["id"]: json.loads(l) for l in open(sample_path, encoding="utf-8")}
ai = [json.loads(l) for l in open(ai_path, encoding="utf-8")]

In [ ]:
rows = []
for a in ai:
    pid = a["id"]
    src = posts.get(pid, {})
    content = a.get("content") or src.get("content")
    human_tldr = a.get("human_tldr") or src.get("summary")
    ai_summary = a.get("summary")

    fh = features_for_post({"content": content, "summary": human_tldr}, with_ner=False)
    fa = features_for_post({"content": content, "summary": ai_summary}, with_ner=False)
    rows.append({
        "id": pid, "subreddit": a.get("subreddit"), "bucket": a.get("_bucket"),
        "content": content, "human": human_tldr, "ai": ai_summary,
        "fp_human": fh["first_person_summary"], "fp_ai": fa["first_person_summary"],
        "q_human": fh["has_question_mark"], "q_ai": fa["has_question_mark"],
        "adv_human": fh["has_advice_marker"], "adv_ai": fa["has_advice_marker"],
        "contain_human": fh["keyword_containment"], "contain_ai": fa["keyword_containment"],
    })
d = pd.DataFrame(rows)

In [ ]:
cos_pq = ROOT / "data" / "processed" / "cosine.parquet"
if cos_pq.exists():
    cos = pd.read_parquet(cos_pq)[["id", "cosine_human", "cosine_ai"]]
    d = d.merge(cos, on="id", how="left")
else:
    d["cosine_human"], _ = pairwise_cosine(d["content"], d["human"])
    d["cosine_ai"], _ = pairwise_cosine(d["content"], d["ai"])

In [ ]:
METRICS = {
    "first_person_density": ("fp_human", "fp_ai"),
    "share_with_question_mark": ("q_human", "q_ai"),
    "share_with_advice_words": ("adv_human", "adv_ai"),
    "cosine_to_post": ("cosine_human", "cosine_ai"),
    "keyword_containment": ("contain_human", "contain_ai"),
}


def agg(frame):
    out = {}
    for name, (hc, ac) in METRICS.items():
        out[(name, "human")] = frame[hc].astype(float).mean()
        out[(name, "ai")] = frame[ac].astype(float).mean()
    return pd.Series(out)


per_sub = d.groupby("subreddit").apply(agg)
overall = agg(d).to_frame("ALL").T
table = pd.concat([per_sub, overall])
table.columns = pd.MultiIndex.from_tuples(table.columns, names=["metric", "source"])
table = table.round(3)
table.to_csv(TAB / "human_vs_ai_by_subreddit.csv")
print("=== Human TL;DR vs AI baseline (means; last row = overall baseline) ===")
with pd.option_context("display.width", 200, "display.max_columns", None):
    print(table)

In [ ]:
subs = list(per_sub.index)
x = np.arange(len(subs)); w = 0.38
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, (hc, ac)) in zip(axes.ravel(), METRICS.items()):
    h = [per_sub.loc[s, (name, "human")] for s in subs]
    a = [per_sub.loc[s, (name, "ai")] for s in subs]
    ax.bar(x - w / 2, h, w, label="human TL;DR", color=COLORS["human"])
    ax.bar(x + w / 2, a, w, label="AI baseline", color=COLORS["ai"], alpha=0.85)
    ax.axhline(overall[(name, "ai")].iloc[0], color=COLORS["ai"], lw=.8, ls=":")
    ax.set_title(name.replace("_", " "), fontsize=10)
    ax.set_xticks(x); ax.set_xticklabels(subs, rotation=45, ha="right", fontsize=7)
    ax.spines[["top", "right"]].set_visible(False)
axes.ravel()[0].legend(frameon=False, fontsize=8)
for ax in axes.ravel()[len(METRICS):]:
    ax.set_visible(False)
fig.suptitle("Human TL;DR vs. AI baseline, by community "
             "(dotted line = overall AI baseline)", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG / "07_human_vs_ai_by_subreddit.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG / "07_human_vs_ai_by_subreddit.svg", bbox_inches="tight")
print("saved 07_human_vs_ai_by_subreddit")